# Modèle LightGBM d'estimation de valeur foncière actuelle d'un bien

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OrdinalEncoder
import lightgbm as lgb
import joblib
import fastparquet as fp

## 1. Chargement du fichier parquet

In [2]:
df = fp.ParquetFile("./data/dvf_merged_since2014_final.parquet").to_pandas()

## 2. Pre-processing

In [3]:
df["date_mutation"] = pd.to_datetime(df["date_mutation"], errors="coerce")

df = df[df["valeur_fonciere"] < 1_000_000].copy()

# Tri par date
df = df.sort_values("date_mutation")

# Target = log(prix)
df["target"] = np.log(df["valeur_fonciere"])

# Sélection des features utiles
features = [
    'date_mutation', 'type_local', 'surface_reelle_bati',
    'nb_pieces_principales', 'code_postal', 'code_departement', 'code_voie',
    'nombre_de_lots', 'surface_m2', 'source_year', 'code_insee', 'lat', 'lon',
    'distance_centre_ville', 'densite_commune', 'altitude_moyenne_commune',
    'population_commune','prix_m2_median_cp','prix_m2_mean_cp','prix_m2_std_cp',
    'surface_m2_median_cp','nb_pieces_mean_cp','transactions_cp','prix_m2_median_cp_last_12m',
    'transactions_cp_last_12m','densite_x_population','densite_population_ratio',
    'densite_altitude_ratio','attractivite_simple','distance_x_densite'
]

X = df[features]
y = df["target"]

## 3. Encodage des variables pour LightGBM

In [4]:
# Donées catégorielles à encoder
categorical_cols = ['type_local', 'code_postal', 'code_departement', 'code_voie']

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[categorical_cols] = encoder.fit_transform(X[categorical_cols])

# Données numériques et date
# datetime → année et mois
X['month_mutation'] = X['date_mutation'].dt.month # l'année est déjà dans source_year
X = X.drop(columns=['date_mutation'])

X['nombre_de_lots'] = pd.to_numeric(X['nombre_de_lots'], errors='coerce').fillna(1)
X['code_insee'] = pd.to_numeric(X['code_insee'], errors='coerce').fillna(0)
X['nb_pieces_principales'] = pd.to_numeric(X['nb_pieces_principales'], errors='coerce').fillna(1)

/tmp/ipykernel_16585/3336854574.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[categorical_cols] = encoder.fit_transform(X[categorical_cols])
/tmp/ipykernel_16585/3336854574.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['month_mutation'] = X['date_mutation'].dt.month # l'année est déjà dans source_year


## 4. Split temporel Train / Test

In [5]:
train_mask = df["date_mutation"] < "2023-11-01"
X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print(f"Taille train : {len(X_train)}")
print(f"Taille test  : {len(X_test)}")

Taille train : 11519249
Taille test  : 1583000


# 5. Le modèle

In [6]:
params = {
    "objective": "regression",
    "metric": "mae",
    "boosting_type": "gbdt",
    "num_leaves": 64,
    "learning_rate": 0.05,
    "n_estimators": 4000,
    "max_depth": -1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "verbose": -1
}

lgb_train = lgb.Dataset(X_train, y_train)
lgb_valid = lgb.Dataset(X_test, y_test)

model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_valid],
    valid_names=["train", "valid"],
    callbacks=[lgb.early_stopping(stopping_rounds=200)]
)

Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[4000]	train's l1: 0.327047	valid's l1: 0.351228


# 6. Prédictions

In [7]:
y_pred_log = model.predict(X_test)
y_pred = np.exp(y_pred_log) 
y_true = np.exp(y_test)

# 7. Evaluation

In [8]:
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("\n--- Performance Globale du modèle ---")
print(f"MAE  : {mae:,.0f} €")
print(f"RMSE : {rmse:,.0f} €")
print(f"R²   : {r2:.4f}")



--- Performance Globale du modèle ---
MAE  : 68,041 €
RMSE : 109,325 €
R²   : 0.5750


In [9]:
# Définition des tranches (en euros)
bins = [0, 200_000, 500_000, 1_000_000]
labels = ["0-200k€", "200k-500k€", "500k-1M€"]

# Attribution tranche par bien selon y_true
price_bins = pd.cut(y_true, bins=bins, labels=labels, right=False)

total_test = len(y_true)

for label in labels:
    mask = (price_bins == label)

    y_true_bin = y_true[mask]
    y_pred_bin = y_pred[mask]

    if len(y_true_bin) == 0:
        print(f"\n--- {label} ---")
        print("Aucun bien dans cette tranche.")
        continue

    proportion = len(y_true_bin) / total_test

    mae_bin = mean_absolute_error(y_true_bin, y_pred_bin)
    rmse_bin = np.sqrt(mean_squared_error(y_true_bin, y_pred_bin))

    print(f"\n--- {label} (proportion : {proportion:.2%}) ---")
    print(f"MAE  : {mae_bin:,.0f} €")
    print(f"RMSE : {rmse_bin:,.0f} €")


--- 0-200k€ (proportion : 53.88%) ---
MAE  : 42,983 €
RMSE : 66,694 €

--- 200k-500k€ (proportion : 38.72%) ---
MAE  : 71,775 €
RMSE : 95,823 €

--- 500k-1M€ (proportion : 7.40%) ---
MAE  : 230,903 €
RMSE : 284,710 €


### Comparaison à une baseline naïve

Comparons maintenant les résultats obtenu avec une simple estimation du bien par rapport au prix médian du mètre carré par code postal, que nous possédons.

In [10]:
df_test = df[~train_mask].copy()

# Baseline naïve
df_test["pred_baseline_median_cp"] = df_test["prix_m2_median_cp"] * df_test["surface_m2"]

# Si surface ou prix_m2_median_cp manquants → fallback sur médiane globale
global_median_price = df[train_mask]["valeur_fonciere"].median()
df_test["pred_baseline_median_cp"] = df_test["pred_baseline_median_cp"].fillna(global_median_price)

y_pred_baseline = df_test["pred_baseline_median_cp"].values

# Scores baseline
mae_base = mean_absolute_error(y_true, y_pred_baseline)
rmse_base = np.sqrt(mean_squared_error(y_true, y_pred_baseline))
r2_base = r2_score(y_true, y_pred_baseline)

print("\n--- BASELINE PRIX_M2_MEDIAN_CP × SURFACE ---")
print(f"MAE  : {mae_base:,.0f} €")
print(f"RMSE : {rmse_base:,.0f} €")
print(f"R²   : {r2_base:.4f}")

print(f"\nGain MAE LightGBM vs Baseline : {mae_base - mae:,.0f} €")


--- BASELINE PRIX_M2_MEDIAN_CP × SURFACE ---
MAE  : 97,209 €
RMSE : 363,174 €
R²   : -3.6904

Gain MAE LightGBM vs Baseline : 29,168 €


# 8. Sauvegarde des modèles

In [11]:
joblib.dump(model, "./models/modele_lightgbm_finalRMSE.pkl")
joblib.dump(encoder, "./models/encoder_categorielles.pkl")

['./models/encoder_categorielles.pkl']